In [7]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path().resolve().parent
DATA_DIR = BASE_DIR / "data"

df = pd.read_csv(DATA_DIR / "food_ingredients_and_allergens.csv")
df.head()

,Food Product,Main Ingredient,Sweetener,Fat/Oil,Seasoning,Allergens,Prediction
0,Almond Cookies,Almonds,Sugar,Butter,Flour,"Almonds, Wheat, Dairy",Contains
1,Almond Cookies,Almonds,Sugar,Butter,Flour,"Almonds, Wheat, Dairy",Contains
2,Chicken Noodle Soup,Chicken broth,NaN,NaN,Salt,"Chicken, Wheat, Celery",Contains
3,Chicken Noodle Soup,Chicken broth,NaN,NaN,Salt,"Chicken, Wheat, Celery",Contains
4,Cheddar Cheese,Cheese,NaN,NaN,Salt,Dairy,Contains


In [8]:
import numpy as np

# Fill NaNs
for c in ["Food Product","Main Ingredient","Sweetener","Fat/Oil","Seasoning","Allergens"]:
    if c in df.columns:
        df[c] = df[c].fillna("")

# Build a text field from columns (this is your ML input)
df["text"] = (
    df["Food Product"].astype(str) + " | " +
    df["Main Ingredient"].astype(str) + " | " +
    df["Sweetener"].astype(str) + " | " +
    df["Fat/Oil"].astype(str) + " | " +
    df["Seasoning"].astype(str)
).str.lower()

# Convert allergens string -> list
def split_allergens(s: str):
    s = str(s).strip()
    if not s:
        return []
    return [x.strip().lower() for x in s.split(",") if x.strip()]

df["allergen_list"] = df["Allergens"].apply(split_allergens)

# Look at unique allergens
all_unique = sorted({a for lst in df["allergen_list"] for a in lst})
all_unique[:30], len(all_unique)


(['alcohol',
  'almonds',
  'anchovies',
  'celery',
  'chicken',
  'cocoa',
  'coconut',
  'dairy',
  'eggs',
  'fish',
  'ghee',
  'milk',
  'mustard',
  'nuts',
  'oats',
  'peanuts',
  'pine nuts',
  'pork',
  'rice',
  'shellfish',
  'soybeans',
  'strawberries',
  'wheat'],
 23)

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer

X_text = df["text"].values
y_lists = df["allergen_list"].values

mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(y_lists)

X_train, X_test, Y_train, Y_test = train_test_split(
    X_text, Y, test_size=0.2, random_state=42
)

mlb.classes_[:20], Y.shape


(array(['alcohol', 'almonds', 'anchovies', 'celery', 'chicken', 'cocoa',
        'coconut', 'dairy', 'eggs', 'fish', 'ghee', 'milk', 'mustard',
        'nuts', 'oats', 'peanuts', 'pine nuts', 'pork', 'rice',
        'shellfish'], dtype=object),
 (399, 23))

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline

allergen_model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1,2), min_df=1, max_features=30000)),
    ("clf", OneVsRestClassifier(LogisticRegression(max_iter=2000)))
])

allergen_model.fit(X_train, Y_train)
print(" Allergen model trained")


 Allergen model trained


C:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 0 is present in all training examples.
  warnings.warn(
C:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 12 is present in all training examples.
  warnings.warn(
C:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 14 is present in all training examples.
  warnings.warn(


In [12]:
from sklearn.metrics import classification_report

Y_pred = allergen_model.predict(X_test)

print(classification_report(
    Y_test, Y_pred,
    target_names=mlb.classes_,
    zero_division=0
))


              precision    recall  f1-score   support

     alcohol       0.00      0.00      0.00         1
     almonds       0.00      0.00      0.00         1
   anchovies       0.00      0.00      0.00         0
      celery       0.00      0.00      0.00         0
     chicken       0.00      0.00      0.00         0
       cocoa       0.00      0.00      0.00         0
     coconut       0.00      0.00      0.00         0
       dairy       1.00      0.54      0.70        46
        eggs       0.00      0.00      0.00         5
        fish       0.00      0.00      0.00         4
        ghee       0.00      0.00      0.00         0
        milk       0.00      0.00      0.00         2
     mustard       0.00      0.00      0.00         1
        nuts       0.00      0.00      0.00         2
        oats       0.00      0.00      0.00         1
     peanuts       0.00      0.00      0.00         0
   pine nuts       0.00      0.00      0.00         0
        pork       0.00    

In [13]:
import joblib
from pathlib import Path

MODEL_DIR = Path().resolve().parent / "model"
MODEL_DIR.mkdir(exist_ok=True)

joblib.dump(allergen_model, MODEL_DIR / "allergen_model.pkl")
joblib.dump(mlb, MODEL_DIR / "allergen_mlb.pkl")

print("Saved:", MODEL_DIR / "allergen_model.pkl")
print("Saved:", MODEL_DIR / "allergen_mlb.pkl")


Saved: C:\Users\User\OneDrive - Sri Lanka Institute of Information Technology\Desktop\Research\PharmaLink\model\allergen_model.pkl
Saved: C:\Users\User\OneDrive - Sri Lanka Institute of Information Technology\Desktop\Research\PharmaLink\model\allergen_mlb.pkl
